In [1]:
!pip install -q --upgrade wandb

import os
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import timm
from torchvision import transforms
from PIL import Image
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics.pairwise import cosine_similarity
from tqdm.notebook import tqdm
import wandb
import random

from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
os.environ["HF_TOKEN"] = user_secrets.get_secret("hf_api")
os.environ["WANDB_API_KEY"] = user_secrets.get_secret("wandb_api")

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.6/25.6 MB 77.2 MB/s eta 0:00:00:00:0100:01
Device: cuda


In [2]:
config = {
    "data_dir": Path("/kaggle/input/competitions/jaguar-re-id"),
    "checkpoint_dir": Path("checkpoints"),
    "megadescriptor_model": "hf-hub:BVRA/MegaDescriptor-L-384",
    "input_size": 384,
    "embedding_dim": 256,
    "hidden_dim": 512,
    "dropout": 0.5,
    "batch_size": 32,
    "learning_rate": 7.67e-4,
    "weight_decay": 1e-4,
    "num_epochs": 25,
    "patience": 7,
    "val_split": 0.2,
    "seed": SEED,
}

config["checkpoint_dir"].mkdir(exist_ok=True)

train_df = pd.read_csv(config["data_dir"] / "train.csv")
label_encoder = LabelEncoder()
train_df['label_encoded'] = label_encoder.fit_transform(train_df['ground_truth'])
num_classes = len(label_encoder.classes_)

train_data, val_data = train_test_split(
    train_df,
    test_size=config["val_split"],
    random_state=config["seed"],
    stratify=train_df['ground_truth']
)

print(f"Train: {len(train_data)} | Val: {len(val_data)} | Classes: {num_classes}")

Train: 1516 | Val: 379 | Classes: 31


In [8]:
# No augmentation (control)
transform_none = transforms.Compose([
    transforms.Resize((config["input_size"], config["input_size"])),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

# Light augmentation
transform_light = transforms.Compose([
    transforms.Resize((config["input_size"], config["input_size"])),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ColorJitter(brightness=0.2, contrast=0.2,
                           saturation=0.1, hue=0.05),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

# Heavy augmentation
transform_heavy = transforms.Compose([
    transforms.Resize((config["input_size"], config["input_size"])),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=15),
    transforms.ColorJitter(brightness=0.4, contrast=0.4,
                           saturation=0.2, hue=0.1),
    transforms.RandomGrayscale(p=0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
    transforms.RandomErasing(p=0.3, scale=(0.02, 0.15)),

])

# Val transform — never augmented
transform_val = transforms.Compose([
    transforms.Resize((config["input_size"], config["input_size"])),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])


class JaguarDataset(Dataset):
    def __init__(self, df, data_dir, transform):
        self.df = df.reset_index(drop=True)
        self.data_dir = data_dir
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = self.data_dir / "train/train" / row['filename']
        try:
            img = Image.open(img_path).convert("RGB")
        except:
            img = Image.fromarray(
                np.zeros((config["input_size"], config["input_size"], 3),
                         dtype=np.uint8))
        return self.transform(img), row['label_encoded']


print("Augmentation strategies defined:")
print("  - none: resize + normalize only (control)")
print("  - light: + horizontal flip + mild color jitter")
print("  - heavy: + rotation + strong color jitter + grayscale + random erasing")

Augmentation strategies defined:
  - none: resize + normalize only (control)
  - light: + horizontal flip + mild color jitter
  - heavy: + rotation + strong color jitter + grayscale + random erasing


In [4]:
class EmbeddingProjection(nn.Module):
    def __init__(self, input_dim, hidden_dim=512, output_dim=256, dropout=0.5):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, output_dim),
            nn.BatchNorm1d(output_dim),
        )

    def forward(self, x):
        return self.network(x)

    def get_embeddings(self, x):
        return F.normalize(self.forward(x), p=2, dim=1)


class CombinedLoss(nn.Module):
    def __init__(self, embedding_dim, num_classes):
        super().__init__()
        self.alpha = 0.3
        self.arcface_weight = nn.Parameter(
            torch.FloatTensor(num_classes, embedding_dim))
        nn.init.xavier_uniform_(self.arcface_weight)
        self.arcface_margin = 0.6
        self.arcface_scale = 48.0
        self.triplet_margin = 0.4

    def arcface_loss(self, embeddings, labels):
        embeddings = F.normalize(embeddings, dim=1)
        weight = F.normalize(self.arcface_weight, dim=1)
        cosine = F.linear(embeddings, weight).clamp(-1+1e-6, 1-1e-6)
        theta = torch.acos(cosine)
        one_hot = torch.zeros_like(cosine)
        one_hot.scatter_(1, labels.view(-1, 1), 1.0)
        output = self.arcface_scale * torch.cos(
            theta + one_hot * self.arcface_margin)
        return F.cross_entropy(output, labels)

    def triplet_loss(self, embeddings, labels):
        embeddings = F.normalize(embeddings, dim=1)
        dist = torch.cdist(embeddings, embeddings, p=2)
        pos_mask = labels.unsqueeze(0) == labels.unsqueeze(1)
        neg_mask = ~pos_mask
        pos_mask.fill_diagonal_(False)
        hardest_pos = (dist * pos_mask.float()).max(dim=1)[0]
        hardest_neg = (dist + 1e6 * (~neg_mask).float()).min(dim=1)[0]
        return F.relu(hardest_pos - hardest_neg + self.triplet_margin).mean()

    def forward(self, embeddings, labels):
        return (self.alpha * self.arcface_loss(embeddings, labels) +
                (1 - self.alpha) * self.triplet_loss(embeddings, labels))


def compute_validation_map(model, megadescriptor, val_loader, val_labels):
    model.eval()
    megadescriptor.eval()
    all_embeddings = []
    with torch.no_grad():
        for images, _ in val_loader:
            images = images.to(device)
            mega_emb = megadescriptor(images)
            proj_emb = model.get_embeddings(mega_emb)
            all_embeddings.append(proj_emb.cpu().numpy())

    all_embeddings = np.vstack(all_embeddings)
    sim_matrix = cosine_similarity(all_embeddings)
    np.fill_diagonal(sim_matrix, -1)

    identity_aps = {}
    for query_idx in range(len(val_labels)):
        query_label = val_labels[query_idx]
        similarities = sim_matrix[query_idx]
        is_match = (val_labels == query_label).astype(int)
        is_match[query_idx] = 0
        sorted_indices = np.argsort(-similarities)
        sorted_matches = is_match[sorted_indices]
        n_positives = sorted_matches.sum()
        if n_positives == 0:
            continue
        cumsum = np.cumsum(sorted_matches)
        precision_at_k = cumsum / np.arange(1, len(sorted_matches) + 1)
        ap = np.sum(precision_at_k * sorted_matches) / n_positives
        if query_label not in identity_aps:
            identity_aps[query_label] = []
        identity_aps[query_label].append(ap)

    return np.mean([np.mean(aps) for aps in identity_aps.values()])


print("Model, loss and validation function ready")

Model, loss and validation function ready


In [5]:
print("Loading MegaDescriptor...")
megadescriptor = timm.create_model(
    config["megadescriptor_model"], pretrained=True)
megadescriptor.eval()
megadescriptor.to(device)

with torch.no_grad():
    dummy = torch.randn(1, 3, config["input_size"],
                        config["input_size"]).to(device)
    megadescriptor_dim = megadescriptor(dummy).shape[1]

print(f"MegaDescriptor loaded | Dim: {megadescriptor_dim}")

Loading MegaDescriptor...


config.json:   0%|          | 0.00/609 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/1.94G [00:00<?, ?B/s]

MegaDescriptor loaded | Dim: 1536


In [10]:
#experiments = [
    {"name": "aug-none", "transform": transform_none,
     "description": "No augmentation (control)"},
    {"name": "aug-light", "transform": transform_light,
     "description": "Light: horizontal flip + mild color jitter"},
    {"name": "aug-heavy", "transform": transform_heavy,
     "description": "Heavy: rotation + strong color jitter + grayscale + random erasing"},
 ]


val_labels = val_data['ground_truth'].values
results = []

for exp in experiments:
    print(f"\n{'='*60}")
    print(f"Running: {exp['name']}")
    print(f"Description: {exp['description']}")
    print(f"{'='*60}")

    torch.manual_seed(SEED)
    np.random.seed(SEED)
    random.seed(SEED)

    train_dataset = JaguarDataset(
        train_data, config["data_dir"], exp["transform"])
    val_dataset = JaguarDataset(
        val_data, config["data_dir"], transform_val)

    train_loader = DataLoader(train_dataset, batch_size=config["batch_size"],
                              shuffle=True, num_workers=2)
    val_loader = DataLoader(val_dataset, batch_size=config["batch_size"],
                            shuffle=False, num_workers=2)

    model = EmbeddingProjection(
        input_dim=megadescriptor_dim,
        hidden_dim=config["hidden_dim"],
        output_dim=config["embedding_dim"],
        dropout=config["dropout"]
    ).to(device)

    loss_fn = CombinedLoss(config["embedding_dim"], num_classes).to(device)

    num_params = sum(p.numel() for p in model.parameters())

    optimizer = torch.optim.AdamW(
        list(model.parameters()) + list(loss_fn.parameters()),
        lr=config["learning_rate"],
        weight_decay=config["weight_decay"]
    )
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', factor=0.5, patience=3)

    wandb.login(key=os.environ["WANDB_API_KEY"])
    wandb.init(
        project="jaguar-reid-mishank",
        name=exp["name"],
        config={
            **config,
            "augmentation": exp["name"],
            "description": exp["description"],
            "loss_function": "combined-arcface-triplet",
            "num_parameters": num_params,
            "num_classes": num_classes,
        }
    )

    best_val_map = 0.0
    best_val_loss = float('inf')
    patience_counter = 0

    for epoch in range(config["num_epochs"]):
        model.train()
        loss_fn.train()
        train_loss = 0

        for images, labels in tqdm(train_loader,
                                   desc=f"Epoch {epoch+1}",
                                   leave=False):
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            with torch.no_grad():
                mega_emb = megadescriptor(images)
            projected = model(mega_emb)
            loss = loss_fn(projected, labels)
            loss.backward()
            optimizer.step()
            train_loss += loss.item()

        train_loss /= len(train_loader)

        model.eval()
        loss_fn.eval()
        val_loss = 0

        with torch.no_grad():
            for images, labels in val_loader:
                images, labels = images.to(device), labels.to(device)
                mega_emb = megadescriptor(images)
                projected = model(mega_emb)
                loss = loss_fn(projected, labels)
                val_loss += loss.item()

        val_loss /= len(val_loader)
        val_map = compute_validation_map(
            model, megadescriptor, val_loader, val_labels)

        scheduler.step(val_loss)
        current_lr = optimizer.param_groups[0]['lr']

        wandb.log({
            "epoch": epoch + 1,
            "train_loss": train_loss,
            "val_loss": val_loss,
            "val_map": val_map,
            "learning_rate": current_lr,
        })

        print(f"Epoch {epoch+1:2d} | "
              f"Train Loss: {train_loss:.4f} | "
              f"Val Loss: {val_loss:.4f} | "
              f"Val mAP: {val_map:.4f}")

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_val_map = val_map
            patience_counter = 0
            torch.save({
                "model_state_dict": model.state_dict(),
                "val_map": best_val_map,
            }, config["checkpoint_dir"] / f"{exp['name']}_best.pth")
        else:
            patience_counter += 1
            if patience_counter >= config["patience"]:
                print(f"Early stopping at epoch {epoch+1}")
                break

    wandb.log({
        "best_val_map": best_val_map,
        "best_val_loss": best_val_loss,
    })
    wandb.finish()

    results.append({
        "augmentation": exp["name"],
        "description": exp["description"],
        "best_val_map": round(best_val_map, 4),
        "best_val_loss": round(best_val_loss, 4),
    })

    print(f"Done: {exp['name']} | Best Val mAP: {best_val_map:.4f}")

print("\n" + "="*60)
print("ALL AUGMENTATION EXPERIMENTS COMPLETE")
print("="*60)
results_df = pd.DataFrame(results)
print(results_df.to_string(index=False))

wandb: WARNING Calling wandb.login() after wandb.init() has no effect.



Running: aug-heavy
Description: Heavy: rotation + strong color jitter + grayscale + random erasing


Epoch 1:   0%|          | 0/48 [00:00<?, ?it/s]

Epoch  1 | Train Loss: 8.3972 | Val Loss: 5.5449 | Val mAP: 0.4624


Epoch 2:   0%|          | 0/48 [00:00<?, ?it/s]

...

Epoch 24 | Train Loss: 1.7510 | Val Loss: 1.5224 | Val mAP: 0.7680


Epoch 25:   0%|          | 0/48 [00:00<?, ?it/s]

Epoch 25 | Train Loss: 1.5304 | Val Loss: 1.5648 | Val mAP: 0.7652


best_val_loss,▁
best_val_map,▁
epoch,▁▁▂▂▂▂▃▃▃▄▄▄▅▅▅▅▆▆▆▇▇▇▇██
learning_rate,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
train_loss,█▆▅▄▄▃▃▃▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁
val_loss,█▅▄▄▃▃▃▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁
val_map,▁▂▄▄▅▅▅▅▆▆▇▆▇▇▇▇▇████████
best_val_loss,1.47037
best_val_map,0.76592
epoch,25
learning_rate,0.00077


Done: aug-heavy | Best Val mAP: 0.7659

ALL AUGMENTATION EXPERIMENTS COMPLETE
augmentation                                                        description  best_val_map  best_val_loss
   aug-heavy Heavy: rotation + strong color jitter + grayscale + random erasing        0.7659         1.4704
